# 04.01 — Case Study: Klasifikasi Sentiment SMSA — DistilBERT vs Groq

**Tujuan**: bandingkan SLM yang baru di-fine-tune (DistilBERT dari modul 03.01) dengan LLM general (Groq llama-3.1-8b) untuk task yang sama: klasifikasi sentiment Bahasa.

**Hipotesis**: SLM fine-tuned **mengalahkan** LLM zero-shot/few-shot di task spesifik ini, dengan latency lebih rendah dan biaya nol.

**Prasyarat**: notebook 03.01 lulus (model fine-tuned tersimpan di `models/finetuned/distilbert-smsa-final/`).

**Sample size**: 100 sample dari test set SMSA (biar Groq tidak rate-limit; juga cukup untuk metric stabil).

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Load test set + sample 100

In [ ]:
from datasets import load_dataset

ds = load_dataset("indonlp/indonlu", "smsa", split="test")
label_names = ds.features["label"].names
print(f"Test set: {len(ds)} | Labels: {label_names}")

SAMPLE_N = 100
test_sample = ds.shuffle(seed=42).select(range(SAMPLE_N))
y_true = test_sample["label"]
texts = test_sample["text"]
print(f"\nUsing {SAMPLE_N} samples.")

## 2. Approach A: DistilBERT fine-tuned (dari modul 03.01)

In [ ]:
import time
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = repo_root / "models" / "finetuned" / "distilbert-smsa-final"
assert MODEL_DIR.exists(), (
    f"Folder model tidak ada: {MODEL_DIR}\n"
    "Jalankan notebook 03.01 dulu untuk fine-tune & save model."
)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.eval()
print(f"DistilBERT fine-tuned loaded ({sum(p.numel() for p in model.parameters()):,} params)")

In [ ]:
import numpy as np

def predict_distilbert(text: str) -> tuple[int, float]:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.softmax(logits, dim=-1)
    return int(probs.argmax()), (time.perf_counter())

# warmup
_ = predict_distilbert("warmup")

preds_distilbert = []
latencies_distilbert = []
for text in texts:
    t0 = time.perf_counter()
    pred, _ = predict_distilbert(text)
    latencies_distilbert.append((time.perf_counter() - t0) * 1000)
    preds_distilbert.append(pred)

print(f"DistilBERT — done. Median latency: {np.median(latencies_distilbert):.0f} ms")

## 3. Approach B: Groq Llama-3.1-8B few-shot

Kita kasih 6 contoh (2 per kelas) di prompt → minta model klasifikasi text baru. Pakai temperature=0 biar deterministik.

In [ ]:
from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

client = groq_client()

FEW_SHOT_EXAMPLES = """
Contoh:
Teks: "Pelayanan ramah, makanan enak, tempat bersih." → positive
Teks: "Saya pesan tapi sampai sekarang belum diantar." → negative
Teks: "Lokasinya strategis, dekat kampus." → neutral
Teks: "Penjualnya tidak sopan dan harganya mahal." → negative
Teks: "Tempatnya biasa saja, nothing special." → neutral
Teks: "Recommended! Pasti balik lagi ke sini." → positive
""".strip()

SYSTEM = (
    "Kamu adalah classifier sentiment Bahasa Indonesia. "
    "Output HANYA satu kata: positive, neutral, atau negative. "
    "Tidak ada penjelasan, tidak ada tanda baca."
)

label2idx = {name: i for i, name in enumerate(label_names)}

def predict_groq(text: str) -> int:
    r = client.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{FEW_SHOT_EXAMPLES}\n\nTeks: \"{text}\" →"},
        ],
        temperature=0, max_tokens=5,
    )
    out = r.choices[0].message.content.strip().lower().split()[0]
    # normalisasi: kadang Groq pakai "positif" Bahasa
    if out.startswith("pos"):
        return label2idx["positive"]
    if out.startswith("neg"):
        return label2idx["negative"]
    return label2idx["neutral"]

In [ ]:
from tqdm.auto import tqdm

preds_groq = []
latencies_groq = []
input_tokens_total = 0
output_tokens_total = 0

for text in tqdm(texts, desc="Groq predict"):
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{FEW_SHOT_EXAMPLES}\n\nTeks: \"{text}\" →"},
        ],
        temperature=0, max_tokens=5,
    )
    latencies_groq.append((time.perf_counter() - t0) * 1000)
    input_tokens_total += r.usage.prompt_tokens
    output_tokens_total += r.usage.completion_tokens

    out = r.choices[0].message.content.strip().lower().split()[0] if r.choices[0].message.content else ""
    if out.startswith("pos"):
        preds_groq.append(label2idx["positive"])
    elif out.startswith("neg"):
        preds_groq.append(label2idx["negative"])
    else:
        preds_groq.append(label2idx["neutral"])

print(f"\nGroq — done. Median latency: {np.median(latencies_groq):.0f} ms")

## 4. Hitung accuracy + biaya + tabel ringkas

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

from utils.benchmark import estimate_cost_usd

acc_d = accuracy_score(y_true, preds_distilbert)
f1_d = f1_score(y_true, preds_distilbert, average="macro")
acc_g = accuracy_score(y_true, preds_groq)
f1_g = f1_score(y_true, preds_groq, average="macro")

cost_groq_total = estimate_cost_usd(input_tokens_total, output_tokens_total)
cost_per_request = cost_groq_total / SAMPLE_N

summary = pd.DataFrame([
    {
        "approach": "DistilBERT fine-tuned (lokal CPU)",
        "accuracy": round(acc_d, 4),
        "f1_macro": round(f1_d, 4),
        "latency_ms_median": round(np.median(latencies_distilbert)),
        "cost_per_request_usd": 0.0,
        "cost_1M_requests_usd": 0.0,
    },
    {
        "approach": "Groq Llama-3.1-8B few-shot",
        "accuracy": round(acc_g, 4),
        "f1_macro": round(f1_g, 4),
        "latency_ms_median": round(np.median(latencies_groq)),
        "cost_per_request_usd": round(cost_per_request, 8),
        "cost_1M_requests_usd": round(cost_per_request * 1_000_000, 2),
    },
])
summary

## 5. Visualisasi: 3 axis (accuracy, latency, cost)

In [ ]:
import matplotlib.pyplot as plt
from utils.plotting import COLORS, setup_style

setup_style()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

names = ["DistilBERT\nfine-tuned", "Groq\nLlama-3.1-8B"]
colors = [COLORS["finetuned"], COLORS["llm"]]

axes[0].bar(names, [acc_d, acc_g], color=colors)
axes[0].set_ylim(0, 1); axes[0].set_title("Accuracy"); axes[0].axhline(0.5, color="red", ls="--", alpha=0.3)
for i, v in enumerate([acc_d, acc_g]): axes[0].text(i, v + 0.02, f"{v:.2%}", ha="center")

axes[1].bar(names, [np.median(latencies_distilbert), np.median(latencies_groq)], color=colors)
axes[1].set_title("Latency median (ms)")
axes[1].set_yscale("log")
for i, v in enumerate([np.median(latencies_distilbert), np.median(latencies_groq)]):
    axes[1].text(i, v * 1.1, f"{v:.0f}", ha="center")

axes[2].bar(names, [0, cost_per_request * 1_000_000], color=colors)
axes[2].set_title("Cost untuk 1M request (USD)")
for i, v in enumerate([0, cost_per_request * 1_000_000]):
    axes[2].text(i, v + max(cost_per_request * 1_000_000, 1) * 0.02, f"${v:.0f}", ha="center")

plt.tight_layout()
plt.show()

## 6. Cek kasus di mana mereka beda

In [ ]:
disagreements = []
for i, text in enumerate(texts):
    if preds_distilbert[i] != preds_groq[i]:
        disagreements.append({
            "text": text[:80] + ("..." if len(text) > 80 else ""),
            "true": label_names[y_true[i]],
            "distilbert": label_names[preds_distilbert[i]],
            "groq": label_names[preds_groq[i]],
        })

print(f"Total disagreement: {len(disagreements)} / {SAMPLE_N}")
pd.DataFrame(disagreements).head(10)

## Refleksi & insight

1. **Accuracy**: biasanya DistilBERT fine-tuned **mengalahkan** Groq zero/few-shot di task ini (selisih 5-15% poin). Itu hasil dari training di domain yang sama.
2. **Latency**: DistilBERT lokal 30-100x lebih cepat dari Groq (no network round-trip, model lebih kecil).
3. **Cost**: di scale 1M request, DistilBERT $0, Groq mungkin $50-200. Nyata.
4. **Tapi**: DistilBERT cuma jago di task ini (sentiment SMSA). Groq jago di **ribuan** task tanpa training tambahan.
5. **Operations**: DistilBERT butuh kamu host server, monitor, retrain saat distribusi data shift. Groq tidak.

**Decision framework** (lengkap di modul 05):
- Klasifikasi specific + volume tinggi + ada labeled data → **SLM fine-tuned**.
- General-purpose / multi-task / low volume / no labeled data → **LLM API**.

## Latihan mandiri

1. Ganti few-shot example di prompt Groq — tambahkan 2 contoh negative yang sarkasme ("Wah hebat sekali pelayanannya, sudah 2 jam menunggu"). Apakah Groq lebih jago handle sarkasme dibanding DistilBERT? Cek di test set.
2. Pakai model Groq yang lebih besar (`llama-3.3-70b-versatile`). Apakah accuracy naik signifikan? Latency & cost-nya?
3. Hitung **biaya impas**: pada berapa request/hari, hosting DistilBERT (mis. $30/bulan VPS) jadi lebih murah dari Groq?

## Lanjut

Case study berikutnya: **summarization** — task generative di mana LLM punya keunggulan natural: [02_summarization_groq_vs_tinyllama.ipynb](02_summarization_groq_vs_tinyllama.ipynb)